# ERP Preprocessing Visualization

**Visual inspection of the full spherical ray-casting and feature-extraction pipeline.**

This notebook walks through every step of the preprocessing pipeline and produces
publication-quality figures suitable for the TCC document:

| Section | What you will see |
|---------|-------------------|
| 1 | 3-D object mesh with centroid |
| 2 | Spherical ERP mapping (θ, φ grid on unit sphere) |
| 3 | Ray-casting hit mask and coverage statistics |
| 4 | All 12 HSDC channels (depth, normals, alignment, gradient) |
| 5 | Surface normal maps encoded as RGB |
| 6 | SWHDC 1-channel depth ERP |
| 7 | First vs. last intersection depth comparison |
| 8 | SWHDC latitude-dependent dilation weights W_n(φ) |
| 9 | Data augmentation effects (rotation, blur, noise) |
| 10 | 3-D sphere textured with the ERP depth map |
| 11 | Per-class gallery (one depth ERP per ModelNet10 class) |
| 12 | Channel statistics and depth distribution |


---
## Imports and Global Style


In [ ]:
import sys
import warnings
from pathlib import Path

# ── project root on sys.path ─────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize, TwoSlopeNorm
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401  (side-effect import)
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

import trimesh

from src.preprocessing.ray_casting import (
    load_mesh, compute_centroid, generate_ray_directions,
    cast_rays, IntersectionData,
)
from src.preprocessing.erp_features import build_hsdc_erp, build_swhdc_erp
from src.preprocessing.augmentation import (
    augment, rotate_erp_3d, gaussian_blur_erp, gaussian_noise_erp,
)

warnings.filterwarnings('ignore')

# ── publication style ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 200,
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

# ── output directory for saved figures ───────────────────────────────────────
FIG_DIR = PROJECT_ROOT / 'figures' / 'preprocessing'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Figures will be saved to: {FIG_DIR}')


---
## Configuration

Set `DATA_ROOT` to your ModelNet10 directory.  
If no real data is available the notebook falls back to a synthetic trimesh object.


In [ ]:
# ── paths — change these to match your local ModelNet directory ──────────────
DATA_ROOT  = PROJECT_ROOT / 'data' / 'raw' / 'modelnet10'
CLASS_NAME = 'bathtub'   # any ModelNet10 class
MESH_IDX   = 10         # sample index within the class

ERP_W, ERP_H = 512, 256
RNG = np.random.default_rng(42)

# ── MODELNET10 classes (for the gallery) ─────────────────────────────────────
MN10_CLASSES = [
    'bathtub', 'bed', 'chair', 'desk', 'dresser',
    'monitor', 'night_stand', 'sofa', 'table', 'toilet',
]

# ── locate or synthesise a mesh ───────────────────────────────────────────────
def find_first_mesh(root: Path, cls: str) -> Path | None:
    for split in ('train', 'test'):
        d = root / cls / split
        if d.exists():
            files = sorted(d.glob('*.off')) + sorted(d.glob('*.obj'))
            if files:
                return files[0]
    return None

MESH_PATH = find_first_mesh(DATA_ROOT, CLASS_NAME)
if MESH_PATH and MESH_IDX > 0:
    all_files = (sorted((DATA_ROOT / CLASS_NAME / 'train').glob('*.off'))
                 + sorted((DATA_ROOT / CLASS_NAME / 'train').glob('*.obj')))
    MESH_PATH = all_files[min(MESH_IDX, len(all_files)-1)] if all_files else MESH_PATH

USE_SYNTHETIC = MESH_PATH is None
if USE_SYNTHETIC:
    print('⚠  ModelNet data not found — using a synthetic mesh for demonstration.')
    # Build a rough chair-like shape from primitives
    seat  = trimesh.creation.box(extents=[1.0, 1.0, 0.12])
    back  = trimesh.creation.box(extents=[1.0, 0.10, 0.80])
    back.apply_translation([0.0, 0.45, 0.46])
    leg   = trimesh.creation.cylinder(radius=0.05, height=0.50)
    offsets = [(-0.40, -0.40), (0.40, -0.40), (-0.40, 0.40), (0.40, 0.40)]
    legs  = []
    for ox, oy in offsets:
        l = leg.copy(); l.apply_translation([ox, oy, -0.31]); legs.append(l)
    mesh_raw = trimesh.util.concatenate([seat, back] + legs)
    mesh_raw.apply_translation(-mesh_raw.centroid)
    OBJECT_LABEL = 'Synthetic chair (no ModelNet data)'
else:
    mesh_raw = load_mesh(MESH_PATH)
    OBJECT_LABEL = f'{CLASS_NAME}  [{MESH_PATH.name}]'

print(f'Object : {OBJECT_LABEL}')
print(f'Vertices: {len(mesh_raw.vertices):,}   Faces: {len(mesh_raw.faces):,}')


---
## Run the Full Preprocessing Pipeline


In [ ]:
centroid  = compute_centroid(mesh_raw)
theta_grid, phi_grid, directions = generate_ray_directions(ERP_W, ERP_H)

print(f'Centroid : ({centroid[0]:.4f}, {centroid[1]:.4f}, {centroid[2]:.4f})')
print(f'Total rays: {ERP_W}×{ERP_H} = {ERP_W*ERP_H:,}')
print('Ray casting …')

idata = cast_rays(mesh_raw, centroid, directions, ERP_W, ERP_H)

erp_hsdc  = build_hsdc_erp(idata)   # (12, H, W)
erp_swhdc = build_swhdc_erp(idata)  # (1, H, W)

hit_rate = idata.hit_mask.mean() * 100
print(f'Hit rate : {hit_rate:.1f}%   max_dist : {idata.max_dist:.5f}')
print(f'HSDC ERP : {erp_hsdc.shape}   SWHDC ERP : {erp_swhdc.shape}')


---
## Section 1 — 3-D Object Mesh

The raw 3-D mesh loaded from the ModelNet dataset.  
Faces are shaded by the **z-component** of their normal vector (Lambertian-like shading).
The **red sphere** marks the area-weighted centroid — the origin of all rays.


In [ ]:
def plot_mesh_3d(mesh: trimesh.Trimesh, centroid: np.ndarray,
                 title: str = '', ax=None):
    """Render a trimesh in a matplotlib 3D axis, shaded by face-normal Z."""
    if ax is None:
        fig = plt.figure(figsize=(5.5, 4.5))
        ax  = fig.add_subplot(111, projection='3d')
    else:
        fig = ax.figure

    verts = mesh.vertices[mesh.faces]           # (N, 3, 3)
    nz    = mesh.face_normals[:, 2]             # (N,) z-component for shading
    shade = np.clip(0.25 + 0.75 * (nz + 1) / 2, 0, 1)
    colors = plt.cm.Blues(shade)               # (N, 4) RGBA

    poly = Poly3DCollection(verts, alpha=0.92, linewidth=0)
    poly.set_facecolor(colors)
    ax.add_collection3d(poly)

    # Centroid
    ax.scatter(*centroid, color='crimson', s=80, zorder=10,
               label='Centroid (camera origin)')

    # Equal aspect ratio
    vmin = mesh.vertices.min(axis=0)
    vmax = mesh.vertices.max(axis=0)
    span = (vmax - vmin).max() / 2
    mid  = (vmax + vmin) / 2
    ax.set_xlim(mid[0]-span, mid[0]+span)
    ax.set_ylim(mid[1]-span, mid[1]+span)
    ax.set_zlim(mid[2]-span, mid[2]+span)

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title or OBJECT_LABEL, pad=8)
    ax.legend(loc='upper left', fontsize=8)
    ax.view_init(elev=20, azim=45)
    return fig, ax


fig, ax = plot_mesh_3d(mesh_raw, centroid)
fig.tight_layout()
fig.savefig(FIG_DIR / '1_mesh_3d.pdf')
plt.show()


In [ ]:
# ── Three views: front / side / top ──────────────────────────────────────────
fig = plt.figure(figsize=(13, 4))
views = [('Front view', 0, 0), ('Side view', 0, 90), ('Top view', 90, 0)]
for i, (label, elev, azim) in enumerate(views, 1):
    ax = fig.add_subplot(1, 3, i, projection='3d')
    plot_mesh_3d(mesh_raw, centroid, title=label, ax=ax)
    ax.view_init(elev=elev, azim=azim)
    ax.legend().set_visible(i == 1)
fig.suptitle(OBJECT_LABEL, fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / '1_mesh_three_views.pdf')
plt.show()


---
## Section 2 — Spherical ERP Mapping (θ, φ Grid)

Every ERP pixel (x, y) corresponds to a **unique 3-D direction** determined by
the inverse spherical camera model:

$$\theta = \frac{2\pi (x + 0.5)}{W}, \quad \phi = \frac{\pi (y + 0.5)}{H}$$

This grid is visualized as latitude/longitude lines on a unit sphere.
The **color** encodes the latitude φ — note how the polar regions (blue)
map to far fewer unique directions than the equator (red), illustrating the
**equirectangular distortion** that HSDC/SWHDC are designed to correct.


In [ ]:
def sphere_erp_diagram(erp_image: np.ndarray | None = None, n_lines: int = 12):
    """Draw a unit sphere with lat/lon grid and optional ERP texture."""
    fig = plt.figure(figsize=(13, 5))
    ax_sphere = fig.add_subplot(121, projection='3d')
    ax_erp    = fig.add_subplot(122)

    # ── sphere surface coloured by latitude (φ) ───────────────────────────────
    H_s, W_s = 64, 128
    ph = np.linspace(1e-4, np.pi - 1e-4, H_s)
    th = np.linspace(0, 2*np.pi, W_s)
    TH, PH = np.meshgrid(th, ph)
    XS = np.sin(PH) * np.cos(TH)
    YS = np.sin(PH) * np.sin(TH)
    ZS = np.cos(PH)

    # colour: latitude φ (0=north pole blue, π/2=equator green, π=south pole red)
    face_c = plt.cm.coolwarm(PH / np.pi)
    ax_sphere.plot_surface(XS, YS, ZS, facecolors=face_c,
                           alpha=0.35, linewidth=0, antialiased=True)

    # ── lat/lon grid lines ────────────────────────────────────────────────────
    for phi_v in np.linspace(np.pi/n_lines, np.pi - np.pi/n_lines, n_lines-1):
        t = np.linspace(0, 2*np.pi, 200)
        ax_sphere.plot(np.sin(phi_v)*np.cos(t), np.sin(phi_v)*np.sin(t),
                       np.full_like(t, np.cos(phi_v)),
                       'k-', lw=0.4, alpha=0.4)
    for theta_v in np.linspace(0, 2*np.pi, n_lines, endpoint=False):
        p = np.linspace(0, np.pi, 200)
        ax_sphere.plot(np.sin(p)*np.cos(theta_v), np.sin(p)*np.sin(theta_v),
                       np.cos(p), 'k-', lw=0.4, alpha=0.4)

    # ── sample rays (sparse grid) ─────────────────────────────────────────────
    rows = np.linspace(8, ERP_H - 8, 6, dtype=int)
    cols = np.linspace(8, ERP_W - 8, 12, dtype=int)
    dirs_2d = directions.reshape(ERP_H, ERP_W, 3)
    for r in rows:
        for c in cols:
            d = dirs_2d[r, c]
            ax_sphere.quiver(0, 0, 0, d[0], d[1], d[2],
                             length=0.95, arrow_length_ratio=0.12,
                             color='navy', alpha=0.6, linewidth=0.7)

    ax_sphere.scatter([0], [0], [0], color='crimson', s=60, zorder=10, label='Camera origin')
    ax_sphere.set_xlim(-1.1, 1.1); ax_sphere.set_ylim(-1.1, 1.1); ax_sphere.set_zlim(-1.1, 1.1)
    ax_sphere.set_xlabel('X'); ax_sphere.set_ylabel('Y'); ax_sphere.set_zlabel('Z')
    ax_sphere.set_title('Unit sphere with ERP ray grid', pad=6)
    ax_sphere.legend(fontsize=8, loc='upper left')
    ax_sphere.view_init(elev=25, azim=40)

    # ── ERP image with lat/lon overlay ────────────────────────────────────────
    if erp_image is not None:
        ax_erp.imshow(erp_image, aspect='auto', cmap='plasma',
                      origin='upper', extent=[0, ERP_W, ERP_H, 0])
        label_tag = 'HSDC d₁ (first depth)'
    else:
        # Show the latitude heat map
        ax_erp.imshow(phi_grid, aspect='auto', cmap='coolwarm',
                      origin='upper', extent=[0, ERP_W, ERP_H, 0])
        label_tag = 'Latitude φ (rad) — distortion map'

    # Overlay lat/lon grid
    for phi_v in np.linspace(0, np.pi, n_lines + 1):
        y_px = phi_v / np.pi * ERP_H
        ax_erp.axhline(y_px, color='white', lw=0.6, alpha=0.5)
    for theta_v in np.linspace(0, ERP_W, n_lines + 1):
        ax_erp.axvline(theta_v, color='white', lw=0.6, alpha=0.5)

    # Annotate poles and equator
    ax_erp.text(5, 8, 'North pole', color='white', fontsize=7, va='top')
    ax_erp.text(5, ERP_H//2, 'Equator', color='white', fontsize=7, va='center')
    ax_erp.text(5, ERP_H - 8, 'South pole', color='white', fontsize=7, va='bottom')
    ax_erp.set_xlabel('x pixel (θ →)')
    ax_erp.set_ylabel('y pixel (φ ↓)')
    ax_erp.set_title(f'ERP image  [{ERP_W} × {ERP_H}]\n{label_tag}', pad=6)

    fig.tight_layout()
    return fig


# Show the latitude distortion map first (no real ERP needed)
fig = sphere_erp_diagram(erp_image=None)
fig.savefig(FIG_DIR / '2_sphere_erp_mapping.pdf')
plt.show()

# Then with the actual depth ERP
fig = sphere_erp_diagram(erp_image=erp_hsdc[0])
fig.savefig(FIG_DIR / '2_sphere_erp_depth.pdf')
plt.show()


---
## Section 3 — Ray-Casting Hit Mask and Coverage

After casting 512 × 256 = 131,072 rays from the centroid, each pixel is classified
as **hit** (the ray intersected the mesh surface) or **miss** (no intersection).
The hit mask directly defines which pixels carry valid feature information.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# ── hit mask ─────────────────────────────────────────────────────────────────
axes[0].imshow(idata.hit_mask, cmap='binary_r', aspect='auto', origin='upper')
axes[0].set_title(f'Hit Mask  ({idata.hit_mask.mean()*100:.1f}% coverage)', pad=6)
axes[0].set_xlabel('x (θ →)'); axes[0].set_ylabel('y (φ ↓)')

# ── raw first-hit depth ───────────────────────────────────────────────────────
im1 = axes[1].imshow(idata.first_dist, cmap='plasma', aspect='auto',
                     origin='upper', vmin=0)
axes[1].set_title('First-hit distance (raw, not normalized)', pad=6)
axes[1].set_xlabel('x (θ →)')
fig.colorbar(im1, ax=axes[1], fraction=0.03, pad=0.02)

# ── depth histogram ───────────────────────────────────────────────────────────
valid_first = idata.first_dist[idata.hit_mask]
valid_last  = idata.last_dist[idata.hit_mask]
axes[2].hist(valid_first, bins=60, alpha=0.65, color='steelblue',
             label=f'd₁  (first hit, μ={valid_first.mean():.3f})')
axes[2].hist(valid_last,  bins=60, alpha=0.65, color='tomato',
             label=f'dₙ  (last hit,  μ={valid_last.mean():.3f})')
axes[2].axvline(idata.max_dist, color='k', ls='--', lw=1.2,
                label=f'max_dist = {idata.max_dist:.4f}')
axes[2].set_xlabel('Distance from centroid'); axes[2].set_ylabel('Pixel count')
axes[2].set_title('Depth distribution (hit pixels only)', pad=6)
axes[2].legend(fontsize=8)

fig.suptitle(OBJECT_LABEL, y=1.02, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_DIR / '3_hit_mask_coverage.pdf')
plt.show()


---
## Section 4 — HSDC 12-Channel ERP

Each object is represented by **12 feature channels** derived from the first and
last ray-mesh intersections (HSDC paper §II-B).

| Row | Channels | Description |
|-----|----------|-------------|
| First hit | 0 d₁, 1–3 N₁, 4 align₁, 5 grad₁ | First intersection features |
| Last hit  | 6 dₙ, 7–9 Nₙ, 10 alignₙ, 11 gradₙ | Last intersection features |


In [ ]:
CHANNEL_META = [
    # (label, colormap, symmetric)
    ('d₁  (first depth)',        'plasma',  False),
    ('Nx₁ (normal x, first)',    'RdBu_r',  True ),
    ('Ny₁ (normal y, first)',    'RdBu_r',  True ),
    ('Nz₁ (normal z, first)',    'RdBu_r',  True ),
    ('align₁ (ray·normal)',      'PiYG',    True ),
    ('grad₁ (gradient mag.)',    'hot',     False),
    ('dₙ  (last depth)',         'plasma',  False),
    ('Nxₙ (normal x, last)',     'RdBu_r',  True ),
    ('Nyₙ (normal y, last)',     'RdBu_r',  True ),
    ('Nzₙ (normal z, last)',     'RdBu_r',  True ),
    ('alignₙ (ray·normal)',      'PiYG',    True ),
    ('gradₙ (gradient mag.)',    'hot',     False),
]

fig, axes = plt.subplots(2, 6, figsize=(18, 6.5),
                          gridspec_kw={'hspace': 0.45, 'wspace': 0.3})

for ch, (ax, (label, cmap, symmetric)) in enumerate(zip(axes.flat, CHANNEL_META)):
    img = erp_hsdc[ch]
    if symmetric:
        vabs = max(abs(img[idata.hit_mask].min()), abs(img[idata.hit_mask].max()))
        vabs = vabs if vabs > 0 else 1.0
        norm = Normalize(vmin=-vabs, vmax=vabs)
    else:
        vmin_val = img[idata.hit_mask].min() if idata.hit_mask.any() else 0
        vmax_val = img[idata.hit_mask].max() if idata.hit_mask.any() else 1
        norm = Normalize(vmin=vmin_val, vmax=vmax_val)

    im = ax.imshow(img, cmap=cmap, norm=norm, aspect='auto', origin='upper')
    ax.set_title(f'Ch {ch}  {label}', fontsize=8.5, pad=4)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

# Row separators
fig.text(0.01, 0.72, 'First\nhit', ha='center', va='center',
         fontsize=10, fontweight='bold', color='steelblue',
         rotation=90)
fig.text(0.01, 0.27, 'Last\nhit',  ha='center', va='center',
         fontsize=10, fontweight='bold', color='tomato',
         rotation=90)

fig.suptitle(f'HSDC — all 12 ERP channels  |  {OBJECT_LABEL}',
             fontsize=12, y=1.01)
fig.savefig(FIG_DIR / '4_hsdc_12channels.pdf')
plt.show()


---
## Section 5 — Surface Normal Maps (RGB Encoding)

Normal vectors **(Nx, Ny, Nz) ∈ [−1, 1]³** are mapped to RGB colours via
**n_rgb = (n + 1) / 2**, producing the vivid normal maps common in computer
graphics.  Red ↔ ±X, Green ↔ ±Y, Blue ↔ ±Z.


In [ ]:
def normals_to_rgb(nx, ny, nz, mask):
    """Encode (Nx,Ny,Nz) ∈ [-1,1] as an RGB image in [0,1]."""
    rgb = np.stack([(nx + 1) / 2,
                    (ny + 1) / 2,
                    (nz + 1) / 2], axis=-1)   # (H, W, 3)
    rgb = np.clip(rgb, 0, 1)
    # Zero out background
    rgb[~mask] = 0.0
    return rgb


rgb_first = normals_to_rgb(erp_hsdc[1], erp_hsdc[2], erp_hsdc[3], idata.hit_mask)
rgb_last  = normals_to_rgb(erp_hsdc[7], erp_hsdc[8], erp_hsdc[9], idata.hit_mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(rgb_first, aspect='auto', origin='upper')
axes[0].set_title('First-hit normal map  (R=Nx, G=Ny, B=Nz)', pad=6)
axes[0].axis('off')
axes[1].imshow(rgb_last,  aspect='auto', origin='upper')
axes[1].set_title('Last-hit normal map   (R=Nx, G=Ny, B=Nz)',  pad=6)
axes[1].axis('off')

# Colour-cube legend
legend_patches = [
    mpatches.Patch(color=(1,0,0), label='+X (red)'),
    mpatches.Patch(color=(0,1,0), label='+Y (green)'),
    mpatches.Patch(color=(0,0,1), label='+Z (blue)'),
    mpatches.Patch(color=(0,0,0), label='No hit'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f'Surface Normal Maps  |  {OBJECT_LABEL}', fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / '5_normal_maps_rgb.pdf')
plt.show()


---
## Section 6 — SWHDC: 1-Channel Depth ERP

The SWHDC pipeline uses only the **last-hit normalized depth** as a single-channel
input (SWHDC paper §IV-A).  This external depth map captures the outer silhouette
of the object as seen from the centroid.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8),
                          gridspec_kw={'wspace': 0.35})

# SWHDC depth
im0 = axes[0].imshow(erp_swhdc[0], cmap='viridis', aspect='auto',
                     origin='upper', vmin=0, vmax=1)
axes[0].set_title('SWHDC: last-hit depth (normalized)', pad=6)
axes[0].axis('off')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02,
             label='Normalized depth')

# HSDC first depth for comparison
im1 = axes[1].imshow(erp_hsdc[0], cmap='viridis', aspect='auto',
                     origin='upper', vmin=0, vmax=1)
axes[1].set_title('HSDC: first-hit depth (normalized)', pad=6)
axes[1].axis('off')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02,
             label='Normalized depth')

# Scatter: first depth vs last depth for hit pixels
d1  = erp_hsdc[0][idata.hit_mask]
dn  = erp_swhdc[0][idata.hit_mask]
axes[2].scatter(d1, dn, s=1, alpha=0.15, c='steelblue')
axes[2].plot([0, 1], [0, 1], 'r--', lw=1.2, label='d₁ = dₙ')
axes[2].set_xlabel('First-hit depth d₁'); axes[2].set_ylabel('Last-hit depth dₙ')
axes[2].set_title('Depth scatter: first vs last hit', pad=6)
axes[2].legend(fontsize=8)
axes[2].set_aspect('equal')

fig.suptitle(f'HSDC vs SWHDC Depth Channels  |  {OBJECT_LABEL}', fontsize=11, y=1.02)
fig.savefig(FIG_DIR / '6_swhdc_depth.pdf')
plt.show()


---
## Section 7 — First vs. Last Intersection: Object Thickness

The **difference dₙ − d₁** (last minus first hit depth) measures how thick the
object is along each ray direction.  Thin regions (edges, flat surfaces face-on)
show near-zero difference; thick interior regions show large positive differences.
This is one of the key advantages of the dual-intersection HSDC pipeline over
single-view representations.


In [ ]:
depth_diff = erp_hsdc[6] - erp_hsdc[0]   # dₙ − d₁, (H, W)
depth_diff[~idata.hit_mask] = np.nan      # mask background

fig, axes = plt.subplots(2, 2, figsize=(13, 7),
                          gridspec_kw={'hspace': 0.45, 'wspace': 0.35})

# d₁
im00 = axes[0,0].imshow(erp_hsdc[0], cmap='plasma', aspect='auto',
                         origin='upper', vmin=0, vmax=1)
axes[0,0].set_title('d₁  — first-hit depth', pad=5)
axes[0,0].axis('off')
fig.colorbar(im00, ax=axes[0,0], fraction=0.046, pad=0.02)

# dₙ
im01 = axes[0,1].imshow(erp_hsdc[6], cmap='plasma', aspect='auto',
                         origin='upper', vmin=0, vmax=1)
axes[0,1].set_title('dₙ  — last-hit depth', pad=5)
axes[0,1].axis('off')
fig.colorbar(im01, ax=axes[0,1], fraction=0.046, pad=0.02)

# depth difference
vmax_d = np.nanmax(np.abs(depth_diff))
im10 = axes[1,0].imshow(depth_diff, cmap='RdYlBu_r', aspect='auto',
                         origin='upper', vmin=0, vmax=vmax_d if vmax_d > 0 else 1)
axes[1,0].set_title('dₙ − d₁  (object thickness along ray)', pad=5)
axes[1,0].axis('off')
fig.colorbar(im10, ax=axes[1,0], fraction=0.046, pad=0.02,
             label='Δ depth (normalized)')

# hit mask
axes[1,1].imshow(idata.hit_mask, cmap='Greens', aspect='auto', origin='upper')
axes[1,1].set_title(f'Hit mask  ({idata.hit_mask.mean()*100:.1f}% coverage)', pad=5)
axes[1,1].axis('off')

fig.suptitle(f'First vs. Last Intersection  |  {OBJECT_LABEL}', fontsize=11, y=1.01)
fig.savefig(FIG_DIR / '7_first_vs_last_depth.pdf')
plt.show()


---
## Section 8 — SWHDC Latitude-Dependent Dilation Weights  W_n(φ)

The SWHDC block corrects ERP distortion by using **different dilation rates at
different latitudes**.  The weight assigned to dilation rate n at row y is:

$$R_\phi = \min\!\left(N,\, \frac{1}{\sin\phi(y)}\right) \quad \text{(Eq. 3)}$$

with linear interpolation between the two nearest integer dilation rates (Eq. 4).
N = 4 covers 96.85 % of the sphere's surface area.

**Key insight:** Near the poles, each ERP pixel spans a very wide horizontal arc
(high R_φ), so a larger dilation rate is used.  At the equator, pixels span a
narrow arc (R_φ = 1), so only the smallest dilation rate contributes.


In [ ]:
N_DILATION = 4

# ── compute W_n(y) for all rows ────────────────────────────────────────────────
phi_rows = np.pi * (np.arange(ERP_H) + 0.5) / ERP_H   # (H,) in (0, π)
sin_phi  = np.sin(phi_rows).clip(min=1e-6)
R_phi    = np.minimum(N_DILATION, 1.0 / sin_phi)       # (H,) Eq. 3

W = np.zeros((N_DILATION, ERP_H), dtype=np.float64)    # (4, H)
for row in range(ERP_H):
    r = R_phi[row]
    r_lo = int(np.floor(r))
    r_hi = int(np.ceil(r))
    r_lo = np.clip(r_lo, 1, N_DILATION)
    r_hi = np.clip(r_hi, 1, N_DILATION)
    if r_lo == r_hi:
        W[r_lo - 1, row] = 1.0
    else:
        W[r_hi - 1, row] = r - r_lo       # Eq. 4 — weight for ceil
        W[r_lo - 1, row] = r_hi - r       # weight for floor

# Sanity check: weights sum to 1
assert np.allclose(W.sum(axis=0), 1.0), 'Weights do not sum to 1'

# ── latitude axis in degrees (from −90° south to +90° north) ─────────────────
lat_deg = 90.0 - phi_rows * 180.0 / np.pi    # convert φ (0…π) to latitude

colors_dil = ['#2166ac', '#4dac26', '#d7191c', '#7b3294']
labels_dil = [f'Dilation r={n}' for n in range(1, N_DILATION+1)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5),
                          gridspec_kw={'wspace': 0.42})

# ── Panel 1: R_φ vs latitude ──────────────────────────────────────────────────
axes[0].plot(lat_deg, R_phi, 'k-', lw=2)
for n in range(1, N_DILATION+1):
    axes[0].axhline(n, color=colors_dil[n-1], ls='--', lw=0.9, alpha=0.7)
axes[0].set_xlabel('Latitude (degrees)')
axes[0].set_ylabel('R_φ  (effective dilation rate)')
axes[0].set_title('R_φ = min(N, 1/sin φ)  [Eq. 3]', pad=6)
axes[0].set_xlim(-90, 90)
axes[0].set_xticks(np.arange(-90, 91, 30))
axes[0].invert_xaxis()

# ── Panel 2: W_n vs latitude (stacked area) ───────────────────────────────────
W_plot = W[:, ::-1].T            # flip so north pole is at left (lat=+90)
lat_flip = lat_deg[::-1]
axes[1].stackplot(lat_flip, W_plot.T, labels=labels_dil, colors=colors_dil, alpha=0.8)
axes[1].set_xlabel('Latitude (degrees)')
axes[1].set_ylabel('Weight W_n(φ)')
axes[1].set_title('Latitude weights W_n(φ)  [Eq. 4] — stacked', pad=6)
axes[1].set_xlim(90, -90)
axes[1].set_xticks(np.arange(-90, 91, 30))
axes[1].legend(loc='upper right', fontsize=8)

# ── Panel 3: dominant dilation rate per row (heatmap on ERP) ─────────────────
dominant_rate = R_phi.reshape(-1, 1) * np.ones((1, ERP_W))   # (H, W)
im = axes[2].imshow(dominant_rate, cmap='plasma', aspect='auto',
                    origin='upper', vmin=1, vmax=N_DILATION,
                    extent=[-180, 180, -90, 90])
axes[2].set_xlabel('Longitude (θ, degrees)')
axes[2].set_ylabel('Latitude (φ, degrees)')
axes[2].set_title('Effective dilation R_φ per ERP row', pad=6)
cbar = fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.02)
cbar.set_ticks([1, 2, 3, 4])
cbar.set_label('R_φ (dilation rate)')

fig.suptitle('SWHDC Latitude-Dependent Dilation Weights', fontsize=12, y=1.02)
fig.savefig(FIG_DIR / '8_swhdc_latitude_weights.pdf')
plt.show()
print(f'N=4 covers {100 * (R_phi < 4).mean():.2f}% of rows (≈96.85% spherical area)')


---
## Section 9 — Data Augmentation Effects

Three augmentation primitives, each applied with **15 % probability** during
training:

1. **3-D rotation** — rotates the spherical signal by remapping pixel directions
   through the inverse rotation matrix (x,y ∈ [0°,15°], z ∈ [0°,45°])
2. **Gaussian blur** — per-channel, σ ∈ [0.1, 2.0]
3. **Gaussian noise** — per-channel, mean ∈ [0, 0.001], σ ∈ [0, 0.03]

The first depth channel (d₁) is used to visualize the effect of each primitive.


In [ ]:
rng_aug = np.random.default_rng(0)

erp_rotated = rotate_erp_3d(erp_hsdc,
                             angle_x_deg=10.0,
                             angle_y_deg=10.0,
                             angle_z_deg=35.0)
erp_blurred = gaussian_blur_erp(erp_hsdc, sigma=1.8)
erp_noisy   = gaussian_noise_erp(erp_hsdc, mean=0.0, std=0.025, rng=rng_aug)
erp_all_aug = augment(erp_hsdc, prob=1.0,
                       rng=np.random.default_rng(7))   # all three at once

versions = [
    ('Original', erp_hsdc),
    ('3-D rotation\n(x=10°, y=10°, z=35°)', erp_rotated),
    ('Gaussian blur\n(σ = 1.8)', erp_blurred),
    ('Gaussian noise\n(σ = 0.025)', erp_noisy),
    ('All three combined\n(prob=1.0, seed=7)', erp_all_aug),
]

fig, axes = plt.subplots(2, 5, figsize=(18, 6),
                          gridspec_kw={'hspace': 0.5, 'wspace': 0.3})

for col, (title, erp_v) in enumerate(versions):
    # Top row: first depth channel (d₁)
    im0 = axes[0, col].imshow(erp_v[0], cmap='plasma', aspect='auto',
                               origin='upper', vmin=0, vmax=1)
    axes[0, col].set_title(title, fontsize=8.5, pad=4)
    axes[0, col].axis('off')

    # Bottom row: normal map (RGB)
    mask_col = erp_v[0] > 0   # approximate hit mask from augmented depth
    rgb = normals_to_rgb(erp_v[1], erp_v[2], erp_v[3], mask_col)
    axes[1, col].imshow(rgb, aspect='auto', origin='upper')
    axes[1, col].axis('off')

fig.text(0.01, 0.73, 'd₁ channel', ha='center', fontsize=10, fontweight='bold',
         color='steelblue', rotation=90)
fig.text(0.01, 0.27, 'Normal map', ha='center', fontsize=10, fontweight='bold',
         color='tomato', rotation=90)

fig.suptitle(f'Data Augmentation Effects  |  {OBJECT_LABEL}', fontsize=12, y=1.02)
fig.savefig(FIG_DIR / '9_augmentation_effects.pdf')
plt.show()


In [ ]:
# ── Difference maps: augmented − original ─────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5),
                          gridspec_kw={'wspace': 0.35})

diff_labels = [
    ('Rotation diff', erp_rotated[0] - erp_hsdc[0]),
    ('Blur diff',     erp_blurred[0] - erp_hsdc[0]),
    ('Noise diff',    erp_noisy[0]   - erp_hsdc[0]),
    ('All-aug diff',  erp_all_aug[0] - erp_hsdc[0]),
]
for ax, (lbl, diff) in zip(axes, diff_labels):
    vmax = max(abs(diff).max(), 1e-6)
    im = ax.imshow(diff, cmap='RdBu_r', aspect='auto', origin='upper',
                   vmin=-vmax, vmax=vmax)
    ax.set_title(lbl, fontsize=9, pad=4)
    ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

fig.suptitle('Augmentation Difference Maps  (d₁ channel, augmented − original)',
             fontsize=11, y=1.02)
fig.savefig(FIG_DIR / '9b_augmentation_diffs.pdf')
plt.show()


---
## Section 10 — 3-D Sphere Textured with ERP Depth

The inverse of the ERP mapping: the normalized depth map is projected back
onto the unit sphere, making the spherical nature of the representation
explicit.  Each point on the sphere is coloured by the depth of the
corresponding ray intersection.


In [ ]:
def plot_sphere_textured(depth_map: np.ndarray, hit_mask: np.ndarray,
                          title: str = '', scale: int = 4):
    """Project a depth ERP onto a 3-D unit sphere.

    Args:
        depth_map: (H, W) normalized depth image.
        hit_mask:  (H, W) bool — background pixels → grey.
        scale:     Downsample factor for performance (default 4 → 128×64).
    """
    from scipy.ndimage import zoom

    H_s = ERP_H // scale
    W_s = ERP_W // scale
    depth_s = zoom(depth_map, 1/scale, order=1)
    mask_s  = zoom(hit_mask.astype(np.float32), 1/scale, order=0) > 0.5

    ph = np.linspace(1e-4, np.pi - 1e-4, H_s)
    th = np.linspace(0, 2*np.pi, W_s, endpoint=False)
    TH, PH = np.meshgrid(th, ph)
    XS = np.sin(PH) * np.cos(TH)
    YS = np.sin(PH) * np.sin(TH)
    ZS = np.cos(PH)

    # Colour: depth for hit pixels, light grey for background
    norm = Normalize(vmin=0, vmax=1)
    face_c = plt.cm.plasma(norm(depth_s))     # (H, W, 4)
    face_c[~mask_s, :3] = 0.85                # grey background
    face_c[~mask_s,  3] = 0.3                 # transparent

    fig = plt.figure(figsize=(14, 5))
    views = [('Perspective', 25, 45), ('Front (XZ)', 0, 0), ('Top (XY)', 90, 0)]
    for i, (vlabel, elev, azim) in enumerate(views):
        ax = fig.add_subplot(1, 3, i+1, projection='3d')
        ax.plot_surface(XS, YS, ZS, facecolors=face_c,
                        linewidth=0, antialiased=False, shade=False)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
        ax.set_title(vlabel, pad=5)
        ax.view_init(elev=elev, azim=azim)
        ax.set_box_aspect([1, 1, 1])

    # Shared colorbar
    sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=fig.axes, fraction=0.015, pad=0.04,
                 label='Normalized depth')
    fig.suptitle(title, fontsize=11, y=1.01)
    return fig


fig = plot_sphere_textured(
    erp_hsdc[6],          # last-hit depth
    idata.hit_mask,
    title=f'ERP depth projected onto unit sphere  |  {OBJECT_LABEL}',
)
fig.savefig(FIG_DIR / '10_sphere_textured.pdf')
plt.show()


In [ ]:
# ── Side-by-side: 3D mesh  ↔  sphere depth  ↔  flat ERP ─────────────────────
fig = plt.figure(figsize=(14, 4.5))

# 3-D mesh
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
plot_mesh_3d(mesh_raw, centroid, title='3-D Mesh', ax=ax1)

# Sphere textured
from scipy.ndimage import zoom as zoom_fn
scale = 4
H_s, W_s = ERP_H // scale, ERP_W // scale
depth_s = zoom_fn(erp_hsdc[6], 1/scale, order=1)
mask_s  = zoom_fn(idata.hit_mask.astype(np.float32), 1/scale, order=0) > 0.5
ph_s = np.linspace(1e-4, np.pi - 1e-4, H_s)
th_s = np.linspace(0, 2*np.pi, W_s, endpoint=False)
TH_s, PH_s = np.meshgrid(th_s, ph_s)
fc = plt.cm.plasma(Normalize()(depth_s))
fc[~mask_s, :3] = 0.88; fc[~mask_s, 3] = 0.2
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax2.plot_surface(np.sin(PH_s)*np.cos(TH_s),
                 np.sin(PH_s)*np.sin(TH_s),
                 np.cos(PH_s),
                 facecolors=fc, linewidth=0, shade=False)
ax2.set_title('Sphere (depth texture)', pad=5)
ax2.view_init(elev=25, azim=45)
ax2.set_xlim(-1.1,1.1); ax2.set_ylim(-1.1,1.1); ax2.set_zlim(-1.1,1.1)

# Flat ERP
ax3 = fig.add_subplot(1, 3, 3)
ax3.imshow(erp_hsdc[6], cmap='plasma', aspect='auto', origin='upper')
ax3.set_title(f'ERP image  [{ERP_W}×{ERP_H}]', pad=5)
ax3.set_xlabel('x pixel (θ →)'); ax3.set_ylabel('y pixel (φ ↓)')

fig.suptitle(f'3-D mesh  →  sphere projection  →  ERP image  |  {OBJECT_LABEL}',
             fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / '10_mesh_sphere_erp_trio.pdf')
plt.show()


---
## Section 11 — Per-Class Gallery

One representative SWHDC depth ERP per ModelNet10 class.
Run this cell only if `DATA_ROOT` points to a valid ModelNet10 directory.


In [ ]:
if not DATA_ROOT.exists():
    print('⚠  DATA_ROOT not found — skipping per-class gallery.')
else:
    fig, axes = plt.subplots(2, 5, figsize=(18, 7.5),
                              gridspec_kw={'hspace': 0.55, 'wspace': 0.3})

    for ax, cls in zip(axes.flat, MN10_CLASSES):
        mpath = find_first_mesh(DATA_ROOT, cls)
        if mpath is None:
            ax.text(0.5, 0.5, f'{cls}\n(not found)',
                    ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue

        try:
            m   = load_mesh(mpath)
            c   = compute_centroid(m)
            _t, _p, dirs = generate_ray_directions(ERP_W, ERP_H)
            d   = cast_rays(m, c, dirs, ERP_W, ERP_H)
            sw  = build_swhdc_erp(d)    # (1, H, W)
            hs  = build_hsdc_erp(d)     # (12, H, W)

            # RGB normal map as background, depth as overlay
            rgb_n = normals_to_rgb(hs[1], hs[2], hs[3], d.hit_mask)
            ax.imshow(rgb_n, aspect='auto', origin='upper', alpha=0.6)
            ax.imshow(sw[0], cmap='plasma', aspect='auto', origin='upper',
                      alpha=0.7, vmin=0, vmax=1)
            ax.set_title(f'{cls}\n{d.hit_mask.mean()*100:.0f}% coverage',
                         fontsize=9, pad=4)
        except Exception as exc:
            ax.text(0.5, 0.5, f'{cls}\nerror:\n{exc}',
                    ha='center', va='center', transform=ax.transAxes,
                    fontsize=7, color='red')
        ax.axis('off')

    fig.suptitle('Per-Class Gallery — ModelNet10 (SWHDC depth + normal overlay)',
                 fontsize=12, y=1.01)
    fig.savefig(FIG_DIR / '11_per_class_gallery.pdf')
    plt.show()


---
## Section 12 — Channel Statistics and Depth Distribution


In [ ]:
ch_short = ['d₁','Nx₁','Ny₁','Nz₁','aln₁','grd₁',
            'dₙ','Nxₙ','Nyₙ','Nzₙ','alnₙ','grdₙ']

# ── Channel statistics over hit pixels ────────────────────────────────────────
stats = {}
for ch in range(12):
    vals = erp_hsdc[ch][idata.hit_mask]
    stats[ch] = {'mean': vals.mean(), 'std': vals.std(),
                 'min': vals.min(),   'max': vals.max()}

fig, axes = plt.subplots(2, 6, figsize=(18, 5),
                          gridspec_kw={'hspace': 0.6, 'wspace': 0.45})

cmaps_hist = ['plasma','RdBu_r','RdBu_r','RdBu_r','PiYG','hot',
              'plasma','RdBu_r','RdBu_r','RdBu_r','PiYG','hot']

for ch, ax in enumerate(axes.flat):
    vals = erp_hsdc[ch][idata.hit_mask]
    n, bins, patches = ax.hist(vals, bins=50, edgecolor='none', density=True)
    # Colour bars by value using the channel's colormap
    cm = plt.get_cmap(cmaps_hist[ch])
    col_norm = Normalize(vmin=bins[0], vmax=bins[-1])
    for patch, bin_l in zip(patches, bins[:-1]):
        patch.set_facecolor(cm(col_norm(bin_l + (bins[1]-bins[0])/2)))
    ax.set_title(f'Ch {ch}  {ch_short[ch]}', fontsize=8.5, pad=3)
    ax.set_xlabel('Value', fontsize=7)
    ax.set_ylabel('Density', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.text(0.97, 0.96, f'μ={stats[ch]["mean"]:.2f}\nσ={stats[ch]["std"]:.2f}',
            ha='right', va='top', transform=ax.transAxes, fontsize=7,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.8))

fig.suptitle(f'HSDC Channel Value Distributions (hit pixels only)  |  {OBJECT_LABEL}',
             fontsize=11, y=1.02)
fig.savefig(FIG_DIR / '12_channel_statistics.pdf')
plt.show()


In [ ]:
# ── Summary statistics table ──────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame({
    'Channel': [f'Ch {ch}  {ch_short[ch]}' for ch in range(12)],
    'Mean':  [f"{stats[ch]['mean']:+.4f}" for ch in range(12)],
    'Std':   [f"{stats[ch]['std']:.4f}"   for ch in range(12)],
    'Min':   [f"{stats[ch]['min']:+.4f}"  for ch in range(12)],
    'Max':   [f"{stats[ch]['max']:+.4f}"  for ch in range(12)],
})
print(f'Object        : {OBJECT_LABEL}')
print(f'Hit rate      : {idata.hit_mask.mean()*100:.2f}%')
print(f'max_dist      : {idata.max_dist:.6f}')
print(f'Mesh faces    : {len(mesh_raw.faces):,}')
print(f'ERP resolution: {ERP_W}×{ERP_H} px\n')
print(df.to_string(index=False))


---
## Section 13 — Bonus: All-in-One Pipeline Summary Figure

A single figure showing the full preprocessing pipeline in one glance:
3-D mesh → hit mask → first depth → last depth → normal map → SWHDC depth.
Suitable as a pipeline diagram figure in the TCC.


In [ ]:
fig = plt.figure(figsize=(18, 4))
gs  = gridspec.GridSpec(1, 6, figure=fig, wspace=0.3)

# 1. 3D mesh
ax0 = fig.add_subplot(gs[0], projection='3d')
plot_mesh_3d(mesh_raw, centroid, title='① 3-D Mesh', ax=ax0)
ax0.set_xlabel(''); ax0.set_ylabel(''); ax0.set_zlabel('')
ax0.legend().set_visible(False)

panels = [
    ('② Hit Mask',        idata.hit_mask.astype(float),  'binary_r', False),
    ('③ First depth d₁',  erp_hsdc[0],                   'plasma',   False),
    ('④ Last depth dₙ',   erp_hsdc[6],                   'plasma',   False),
    ('⑤ Normal map (RGB)', None,                          None,       False),
    ('⑥ SWHDC depth',     erp_swhdc[0],                  'viridis',  False),
]

for col, (title, img, cmap, _sym) in enumerate(panels, 1):
    ax = fig.add_subplot(gs[col])
    if title.startswith('⑤'):   # RGB normal map
        ax.imshow(rgb_first, aspect='auto', origin='upper')
    else:
        ax.imshow(img, cmap=cmap, aspect='auto', origin='upper')
    ax.set_title(title, fontsize=9, pad=4)
    ax.axis('off')

fig.suptitle(f'ERP Preprocessing Pipeline Summary  |  {OBJECT_LABEL}',
             fontsize=12, y=1.05)
fig.savefig(FIG_DIR / '13_pipeline_summary.pdf')
plt.show()
print(f'\nAll figures saved to: {FIG_DIR}')
